# House Price Prediction with XGBoost (Regression)

**Goal:** Predict house prices using property features (e.g., sqft, grade, location features).

**What you’ll see in this notebook:**
- A clear end-to-end ML workflow (data loading → EDA → baseline → XGBoost → tuning → evaluation)
- A reproducible pipeline mindset (train/validation split, metrics, random seeds)
- Space for experimentation (feature engineering + hyperparameter search playground)

**Dataset:** Loaded from a public CSV used in the course:
- `housing.csv` from BYUI CSE450

## Table of contents
1. Setup
2. Load data
3. Quick EDA
4. Train/validation split
5. Baseline model
6. XGBoost model
7. Model interpretation
8. Experimentation playground
9. Conclusions & next steps


In [ ]:
# 1) Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.dummy import DummyRegressor

# XGBoost
# If running locally, install with: pip install xgboost
from xgboost import XGBRegressor

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def regression_report(y_true, y_pred, label="Model"):
    return {
        "model": label,
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred))
    }

In [ ]:
# 2) Load data
DATA_URL = "https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv"
df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
df.head()

## 3) Quick EDA

We inspect the target distribution and a few key predictors.

**Target:** `price`

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df['price'], bins=50, kde=True)
plt.title('Target Distribution: price')
plt.xlabel('price')
plt.ylabel('count')
plt.show()

plt.figure(figsize=(7,4))
sns.histplot(np.log1p(df['price']), bins=50, kde=True)
plt.title('Target Distribution (log1p): price')
plt.xlabel('log1p(price)')
plt.ylabel('count')
plt.show()

### A note on modeling the target

House prices are often right-skewed. A common technique is to model `log(price)` and then transform back.
In this notebook we will:
- start with modeling `price` directly (baseline)
- then optionally evaluate a `log1p(price)` approach in the experimentation section


## 4) Train/validation split

We keep a holdout set to estimate real-world performance.

In [ ]:
TARGET = 'price'

# Drop columns that are identifiers or date-like values.
# Keeping 'date' often requires parsing; for a clean baseline we drop it.
drop_cols = ['id', 'date']

X = df.drop(columns=[TARGET] + [c for c in drop_cols if c in df.columns])
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Val:", X_val.shape)
X_train.head()

## 5) Baseline model

A baseline is essential to justify using a more complex model. We'll start with:
- `DummyRegressor(strategy='mean')`


In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_val)

baseline_metrics = regression_report(y_val, pred_base, label='DummyRegressor(mean)')
baseline_metrics

## 6) XGBoost model (first strong model)

XGBoost is often a strong choice for tabular regression.

We start with a reasonable configuration and then improve it in the experimentation section.


In [ ]:
xgb = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    objective='reg:squarederror'
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

pred_xgb = xgb.predict(X_val)
xgb_metrics = regression_report(y_val, pred_xgb, label='XGBRegressor(initial)')

pd.DataFrame([baseline_metrics, xgb_metrics]).sort_values('rmse')

### Residual analysis

We visualize prediction errors to spot bias (e.g., under-predicting expensive houses).

In [ ]:
residuals = y_val - pred_xgb

plt.figure(figsize=(7,4))
sns.scatterplot(x=pred_xgb, y=residuals, alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuals vs Predicted')
plt.xlabel('Predicted price')
plt.ylabel('Residual (y - y_pred)')
plt.show()

plt.figure(figsize=(7,4))
sns.histplot(residuals, bins=50, kde=True)
plt.title('Residual distribution')
plt.xlabel('Residual')
plt.show()

## 7) Model interpretation (feature importance)

We inspect global feature importance from XGBoost.

> For production-grade interpretation, you can add permutation importance or SHAP in the experimentation section.

In [ ]:
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

importances.head(15)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=importances.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importances (XGBoost)')
plt.xlabel('importance')
plt.ylabel('feature')
plt.show()

# 8) Experimentation Playground (recommended)

This section is intentionally designed as a sandbox:
- Try feature engineering
- Try modeling `log1p(price)`
- Try hyperparameter search

The goal is to show experimentation *without* losing the narrative above.


## 8.1 Optional: Log-target modeling

Sometimes it improves RMSE and reduces skew issues. We fit XGBoost on `log1p(price)` and then invert.


In [ ]:
USE_LOG_TARGET = False

if USE_LOG_TARGET:
    y_train_log = np.log1p(y_train)
    y_val_log = np.log1p(y_val)

    xgb_log = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        objective='reg:squarederror'
    )

    xgb_log.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], verbose=False)
    pred_log = xgb_log.predict(X_val)
    pred_price = np.expm1(pred_log)

    log_metrics = regression_report(y_val, pred_price, label='XGBRegressor(log1p target)')
    display(pd.DataFrame([baseline_metrics, xgb_metrics, log_metrics]).sort_values('rmse'))
else:
    print("Log-target experiment disabled. Set USE_LOG_TARGET=True to run.")

## 8.2 Optional: Hyperparameter tuning (lightweight)

Below is a lightweight manual tuning template.

> If you want, we can later upgrade this to RandomizedSearchCV and CV folds, but this already demonstrates experimentation cleanly.


In [ ]:
RUN_MANUAL_TUNING = False

if RUN_MANUAL_TUNING:
    candidates = [
        dict(max_depth=4, min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
        dict(max_depth=6, min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
        dict(max_depth=8, min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
        dict(max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
        dict(max_depth=6, min_child_weight=1, subsample=0.6, colsample_bytree=0.8, reg_lambda=1.0),
        dict(max_depth=6, min_child_weight=1, subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0),
        dict(max_depth=6, min_child_weight=1, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0),
    ]

    results = []
    for params in candidates:
        m = XGBRegressor(
            n_estimators=2000,
            learning_rate=0.03,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            objective='reg:squarederror',
            **params
        )
        m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        p = m.predict(X_val)
        results.append({**params, **regression_report(y_val, p, label='candidate')})

    results_df = pd.DataFrame(results).sort_values('rmse')
    display(results_df.head(10))
else:
    print("Manual tuning disabled. Set RUN_MANUAL_TUNING=True to run.")

# 9) Conclusions & Next Steps

### Key takeaways
- A baseline model provides a minimum performance reference.
- XGBoost usually yields strong performance on tabular regression problems.
- Feature importance highlights which property characteristics most affect the prediction.

### Next steps (to make it even more portfolio-grade)
- Use **cross-validation** and **RandomizedSearchCV** for more robust tuning.
- Add **SHAP** for clearer interpretability.
- Add a **final test set** and a **model export** step (joblib) for deployment readiness.
